# qGridX quickstart

This notebook takes a transmission planning instance from real power flow
physics to a buildable storage plan on six qubits, then screens that plan
against every single line outage.

It runs in a few minutes on the qBraid free tier. Nothing here needs a real
quantum device; the device campaign lives in `02_hardware.ipynb`.

In [ ]:
%pip install -q -e ..
import qgridx
print("qGridX", qgridx.__version__)

## 1. Build a planning instance from grid physics

The instance is not synthetic. Locational marginal prices come from a DC
optimal power flow on an IEEE test system, and the quadratic coupling comes
from shared congestion exposure between candidate buses.

In [ ]:
from qgridx.problems import load_registry, rebuild_instance

reg = load_registry()
print(f"{len(reg)} certified instances across {reg.grid.nunique()} grids")

row = reg[reg.grid == "IEEE-118"].iloc[0]
inst = rebuild_instance(row)
print(f"{row.instance_id}: m={inst.m} decisions, "
      f"B={inst.B} candidate buses, L={inst.L} capacity tiers")
print(f"certified optimum: {row.optimum_value:.4f}")

## 2. How many qubits does that need?

An `n`-qubit register carries `3 * C(n, k)` decisions, so the register grows
roughly logarithmically in the decision count rather than linearly.

In [ ]:
from qgridx.encoding import max_capacity
from qgridx.pipeline import smallest_register

n = smallest_register(inst.m, k=2)
print(f"{inst.m} decisions fit on {n} qubits "
      f"(capacity {max_capacity(n, 2)}), against {inst.m} qubits "
      f"for a one-qubit-per-decision mapping")

for nq, k in [(6, 2), (7, 3), (15, 5), (14, 5)]:
    print(f"  n={nq:2d}, k={k}:  capacity {max_capacity(nq, k):>5d}")

## 3. Solve it

`solve` assigns decisions to Pauli strings, trains the generator on the cost
of the plans its circuits decode to, and repairs the resulting correlation
signs into a budget-feasible plan.

The evaluation budget is reduced here so the cell finishes quickly. Reported
results use 10,000 evaluations for every arm.

In [ ]:
from qgridx.pipeline import solve

plan = solve(inst, evals=1500, seed=101)
print(f"cost      {plan.cost:.4f}   (certified optimum {row.optimum_value:.4f})")
print(f"qubits    {plan.n_qubits}")
print(f"gates     {plan.n_gates}")
print(f"sited     {plan.n_sites} buses, {plan.sited_mw:.0f} MW")
print(f"tiers     {plan.levels}")

## 4. Is the plan secure under N-1?

Open every branch in turn, rebuild the post-outage transfer factors, and
solve a security-constrained power flow that is allowed to shed load. Doing
it twice, with and without the sited storage on identical outages, makes the
resilience contribution a measured difference rather than a claim.

In [ ]:
import numpy as np
from qgridx.grid import (compute_ptdf_outage, solve_dcopf_with_shedding,
                         plan_to_bus_mw)
from qgridx.grid.cases import get_case
from qgridx.problems.registry import load_vector

case = get_case(row.grid)
load = load_vector(row, inst.load_scale)
storage = plan_to_bus_mw(inst, plan.x)

def sweep(plan_mw):
    unserved, secure = [], []
    for br in range(case.BRANCH.shape[0]):
        try:
            ptdf = compute_ptdf_outage(case, br)
        except np.linalg.LinAlgError:
            continue                      # outage splits the network
        r = solve_dcopf_with_shedding(load, ptdf, case, extra_gen_bus_mw=plan_mw)
        unserved.append(r.unserved_mw if r.feasible else load.sum())
        secure.append(r.feasible and r.unserved_mw < 1e-6)
    return float(np.mean(unserved)), float(np.mean(secure))

eue_no, sec_no = sweep(None)
eue_yes, sec_yes = sweep(storage or None)
print(f"storage sited at {storage}")
print(f"without storage:  {100*sec_no:5.1f}% of outages served in full, "
      f"EUE {eue_no:.3f} MW")
print(f"with storage:     {100*sec_yes:5.1f}% of outages served in full, "
      f"EUE {eue_yes:.3f} MW")

## 5. Compare against the archived results

Every reported number ships with the package, so a fresh run can be checked
against the paper without rerunning hours of compute.

In [ ]:
import pandas as pd
from qgridx.utils.paths import results_dir

g1 = pd.read_csv(results_dir() / "doe_phase3" / "g1_resilience.csv")
summary = (g1.groupby(["grid", "ai_load_mw"])
             .agg(secure_no=("secure_frac_no_storage", "mean"),
                  secure_with=("secure_frac_with_storage", "mean"),
                  eue_cut_pct=("eue_reduction_pct", "mean"))
             .round(3))
summary

## Next

- `02_hardware.ipynb` runs the measurement protocol on a real device.
- `qgridx --list` shows every reproduction command.
- `qgridx benchmark --smoke` checks the full pipeline in about a minute.